In [1]:
import open3d as o3d
import numpy as np
import copy
mesh=o3d.io.read_triangle_mesh("cup.obj")
pcd=mesh.sample_points_uniformly(number_of_points=20000)
pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamKNN(knn=30))
pcd.orient_normals_consistent_tangent_plane(k=30)
fpfh=o3d.pipelines.registration.compute_fpfh_feature(
    pcd,o3d.geometry.KDTreeSearchParamKNN(knn=30))


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
P = np.asarray(pcd.points)
N = np.asarray(pcd.normals)
radial = P[:, :2] / np.linalg.norm(P[:, :2], axis=1, keepdims=True)
dot = (N[:, :2] * radial).sum(axis=1)
print((dot > 0).mean())

0.7779


In [3]:
P = np.asarray(pcd.points)
N = np.asarray(pcd.normals)
radial = P[:, :2] / np.linalg.norm(P[:, :2], axis=1, keepdims=True)
dot = (N[:, :2] * radial).sum(axis=1)
N[dot < 0] *= -1
pcd.normals = o3d.utility.Vector3dVector(N)
dot = (N[:, :2] * radial).sum(axis=1)
print((dot > 0).mean())

0.99725


In [4]:
pcd1=copy.deepcopy(pcd)
theta=np.radians(30)
R_gt=np.array([
    [1,0,0],
    [0,np.cos(theta),-np.sin(theta)],
    [0,np.sin(theta),np.cos(theta)]
    ])
t_gt=np.array([20.0,-10.0,15.0])
T_gt=np.eye(4)
T_gt[:3,:3]=R_gt
T_gt[:3,3]=t_gt
pcd1.transform(T_gt)

PointCloud with 20000 points.

In [5]:
def prep(pcd):
    pcd.estimate_normals(o3d.geometry.KDTreeSearchParamKNN(knn=30))
    pcd.orient_normals_consistent_tangent_plane(k=30)
    P=np.asarray(pcd.points);N=np.asarray(pcd.normals)
    radial=P[:,:2]/np.linalg.norm(P[:,:2],axis=1,keepdims=True)
    N[(N[:,:2]*radial).sum(1)<0]*=-1
    pcd.normals=o3d.utility.Vector3dVector(N)
    f=o3d.pipelines.registration.compute_fpfh_feature(pcd,o3d.geometry.KDTreeSearchParamKNN(knn=30))
    return pcd,f
pcd,f1=prep(pcd)
pcd1,f2=prep(pcd1)

In [6]:
pcd2 = mesh.sample_points_uniformly(number_of_points=20000)
P2 = np.asarray(pcd2.points) @ R_gt.T + t_gt
P2 += np.random.default_rng(0).normal(0, 0.3, P2.shape)
pcd2 = pcd2.voxel_down_sample(1.0)
pcd2,f3=prep(pcd2)

In [7]:
res=o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    pcd,pcd1,f1,f2,True,
    max_correspondence_distance=3.0,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
    ransac_n=3,
    checkers=[o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
              o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(3.0)],
    criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(100000,0.999))

In [8]:
R_est = res.transformation[:3, :3]
t_est = res.transformation[:3, 3]
ang = np.degrees(np.arccos(np.clip((np.trace(R_est.T @ R_gt) - 1) / 2, -1, 1)))
print("角度误差", ang, "度")
print("平移误差", np.linalg.norm(t_est - t_gt), "mm")

角度误差 0.0 度
平移误差 2.040878629308206e-14 mm


In [9]:
res1=o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    pcd,pcd2,f1,f3,True,
    max_correspondence_distance=3.0,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
    ransac_n=3,
    checkers=[o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
              o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(3.0)],
    criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(100000,0.999))

In [10]:
R_est1 = res1.transformation[:3, :3]
t_est1 = res1.transformation[:3, 3]
ang = np.degrees(np.arccos(np.clip((np.trace(R_est1.T @ R_gt) - 1) / 2, -1, 1)))
print("角度误差", ang, "度")
print("平移误差", np.linalg.norm(t_est1 - t_gt), "mm")

角度误差 61.720053054299505 度
平移误差 26.327352243508162 mm


In [14]:
pcd.paint_uniform_color([1, 0, 0])
pcd2.paint_uniform_color([0, 1, 0])
o3d.visualization.draw_geometries([pcd, pcd2])

In [12]:
res_icp = o3d.pipelines.registration.registration_icp(
    pcd, pcd2, 
    max_correspondence_distance=50.0, 
    init=res1.transformation,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPlane()
)
R_est_icp = res_icp.transformation[:3, :3]
t_est_icp = res_icp.transformation[:3, 3]
ang_icp = np.degrees(np.arccos(np.clip((np.trace(R_est_icp.T @ R_gt) - 1) / 2, -1, 1)))

print("ICP精修后的角度误差", ang_icp, "度")
print("ICP精修后的平移误差", np.linalg.norm(t_est_icp - t_gt), "mm")

ICP精修后的角度误差 61.33303412550646 度
ICP精修后的平移误差 26.936548641077746 mm


In [13]:
pcd_aligned = copy.deepcopy(pcd)
pcd_aligned.transform(res_icp.transformation)
evaluation = o3d.pipelines.registration.evaluate_registration(
    pcd_aligned, pcd2, 2.0
)

print(f"内点率 (Fitness): {evaluation.fitness * 100:.2f}%")
print(f"点云配准误差 (RMSE): {evaluation.inlier_rmse:.4f} mm")

内点率 (Fitness): 99.46%
点云配准误差 (RMSE): 0.8674 mm


In [15]:
zv = np.array([0., 0., 1.])
ang = np.degrees(np.arccos(np.clip(
    abs((R_est @ zv) @ (R_gt @ zv)), -1, 1)))
print("轴误差", ang)

轴误差 0.0
